In [ ]:
!git clone https://github.com/Atharvy700/SP-25.git


Cloning into 'SP-25'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 58 (delta 7), reused 48 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (58/58), 8.44 MiB | 17.08 MiB/s, done.
Resolving deltas: 100% (7/7), done.


In [ ]:
%cd SP-25

/content/SP-25


In [ ]:
!ls -R

.:
bert_finetune.py  make_datasets.py  reflexivity.py  trans_eq_data
bert_noshot.py	  qwen_finetune.py  symmetry.py
datasets	  qwen_noshot.py    synthesis.py

./datasets:
reflexive  symmetric  transitive

./datasets/reflexive:
refl_test_iid.jsonl  refl_test_long.jsonl  refl_train.jsonl  refl_valid.jsonl

./datasets/symmetric:
sym_test_iid.jsonl  sym_test_long.jsonl  sym_train.jsonl  sym_valid.jsonl

./datasets/transitive:
eq_chain  mixed  negation  quantifiers	relations

./datasets/transitive/eq_chain:
trans_eq_test_iid.jsonl   trans_eq_train.jsonl
trans_eq_test_long.jsonl  trans_eq_valid.jsonl

./datasets/transitive/mixed:
mixed_test.jsonl  mixed_train.jsonl  mixed_valid.jsonl

./datasets/transitive/negation:
trans_neg_test.jsonl  trans_neg_train.jsonl  trans_neg_valid.jsonl

./datasets/transitive/quantifiers:
trans_quant_test.jsonl	trans_quant_train.jsonl  trans_quant_valid.jsonl

./datasets/transitive/relations:
trans_rel_test.jsonl  trans_rel_train.jsonl  trans_rel_valid.jsonl

./t

In [ ]:
import json, gc, torch, numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
TEST_PATH = "trans_eq_data/test.json"
MAX_LENGTH = 256

YES = {"yes","true","1"}
NO = {"no","false","0"}

def normalize(x):
    if isinstance(x,bool): return "yes" if x else "no"
    if isinstance(x,int): return "yes" if x==1 else "no"
    s=str(x).lower()
    if s in YES: return "yes"
    if s in NO: return "no"
    return None

def build_prompt(text):
    return (
        "You are a logical reasoning assistant.\n"
        "Answer with exactly one word.\n\n"
        f"{text}\n\n"
        "Answer only yes or no:"
    )

@torch.inference_mode()
def predict(model, tok, prompt):
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to("cuda")
    out = model(**inputs)
    logits = out.logits[0, -1]
    yes_id = tok("yes", add_special_tokens=False).input_ids[0]
    no_id  = tok("no",  add_special_tokens=False).input_ids[0]
    return "yes" if logits[yes_id] > logits[no_id] else "no"

def main():
    if not torch.cuda.is_available():
        raise RuntimeError("Enable GPU: Runtime → Change runtime type → GPU")

    gc.collect()
    torch.cuda.empty_cache()

    with open(TEST_PATH) as f:
        data = json.load(f)

    gold = [normalize(x["label"]) for x in data]
    prompts = [build_prompt(x["text"]) for x in data]

    print("Loading tokenizer...")
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    tok.pad_token = tok.eos_token

    print("Loading model (FP16, no quantization)...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    print("Running inference...")
    preds = [predict(model, tok, p) for p in prompts]

    valid = [(p,g) for p,g in zip(preds,gold) if g is not None]
    acc = np.mean([p==g for p,g in valid])

    print("\n==============================")
    print("Model:", MODEL_NAME)
    print("Zero-shot accuracy:", round(acc,4))
    print("==============================\n")

if __name__ == "__main__":
    main()


Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Running inference...

Model: microsoft/Phi-3-mini-4k-instruct
Zero-shot accuracy: 0.7623



# finetuned

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json, gc, torch
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model

# ===================== CONFIG =====================
MODEL_NAME  = "microsoft/Phi-3-mini-4k-instruct"

TRAIN_PATH  = "trans_eq_data/train.json"
VALID_PATH  = "trans_eq_data/valid.json"
TEST_PATH   = "trans_eq_data/test.json"

OUTPUT_DIR  = "./phi3_lora_out"

MAX_LENGTH  = 256

EPOCHS      = 3
LR          = 2e-4
BATCH_SIZE  = 1
GRAD_ACCUM  = 8

# rank-r tuning
LORA_R      = 16
LORA_ALPHA  = 32
LORA_DROPOUT= 0.05
# ==================================================

YES = {"yes","true","1","y"}
NO  = {"no","false","0","n"}

def normalize_label(x):
    if isinstance(x, bool): return "yes" if x else "no"
    if isinstance(x, int):  return "yes" if x == 1 else "no"
    s = str(x).strip().lower()
    if s in YES: return "yes"
    if s in NO:  return "no"
    raise ValueError(f"Bad label: {x}")

def build_prompt(text):
    return (
        "You are a logical reasoning assistant.\n"
        "Answer with exactly one word.\n\n"
        f"{text}\n\n"
        "Answer only yes or no:"
    )

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def make_sft_rows(data):
    """
    We create:
      prompt_text = "... Answer only yes or no:"
      target = " yes" or " no"
    Then we train with labels masked so only target tokens contribute to loss.
    """
    rows = []
    for ex in data:
        prompt = build_prompt(ex["text"])
        target = " " + normalize_label(ex["label"])
        rows.append({"prompt": prompt, "target": target})
    return rows

def tokenize_with_answer_mask(rows, tokenizer):
    """
    Create input_ids for prompt+target, but labels are -100 for prompt tokens,
    and only target tokens are supervised.
    """
    prompt_ids = tokenizer(rows["prompt"], add_special_tokens=False)["input_ids"]
    target_ids = tokenizer(rows["target"], add_special_tokens=False)["input_ids"]

    input_ids = (prompt_ids + target_ids)[:MAX_LENGTH]
    attention_mask = [1] * len(input_ids)

    labels = ([-100] * len(prompt_ids) + target_ids)[:MAX_LENGTH]

    # pad
    pad_id = tokenizer.pad_token_id
    if len(input_ids) < MAX_LENGTH:
        pad_len = MAX_LENGTH - len(input_ids)
        input_ids += [pad_id] * pad_len
        attention_mask += [0] * pad_len
        labels += [-100] * pad_len

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

@torch.inference_mode()
def eval_yesno_accuracy(model, tokenizer, test_data):
    def predict_one(text):
        prompt = build_prompt(text)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to("cuda")
        out = model(**inputs)
        logits = out.logits[0, -1]

        yes_id = tokenizer("yes", add_special_tokens=False).input_ids[0]
        no_id  = tokenizer("no",  add_special_tokens=False).input_ids[0]
        return "yes" if logits[yes_id] > logits[no_id] else "no"

    gold = [normalize_label(x["label"]) for x in test_data]
    preds = [predict_one(x["text"]) for x in test_data]
    acc = np.mean([p == g for p, g in zip(preds, gold)])
    return float(acc)

def main():
    if not torch.cuda.is_available():
        raise RuntimeError("Enable GPU: Runtime → Change runtime type → GPU")

    gc.collect()
    torch.cuda.empty_cache()

    print("Loading data...")
    train_rows = make_sft_rows(load_json(TRAIN_PATH))
    valid_rows = make_sft_rows(load_json(VALID_PATH))
    test_data  = load_json(TEST_PATH)

    train_ds = Dataset.from_list(train_rows)
    valid_ds = Dataset.from_list(valid_rows)

    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Tokenizing with answer-only supervision...")
    train_ds = train_ds.map(lambda r: tokenize_with_answer_mask(r, tokenizer), remove_columns=["prompt","target"])
    valid_ds = valid_ds.map(lambda r: tokenize_with_answer_mask(r, tokenizer), remove_columns=["prompt","target"])

    print("Loading base model (FP16, no quantization)...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
    )

    # Recommended for memory + speed
    model.gradient_checkpointing_enable()
    model.config.use_cache = False

    print("Applying LoRA (rank-r tuning)...")
    lora_cfg = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj"],
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

    args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        logging_steps=20,
        eval_strategy="steps",     # (new transformers name)
        eval_steps=200,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=2,
        fp16=True,
        report_to="none",
        optim="adamw_torch",
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        tokenizer=tokenizer,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )

    print("Training...")
    trainer.train()

    print("Saving adapter...")
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

    print("\nEvaluating on test set (yes/no accuracy)...")
    model.eval()
    acc = eval_yesno_accuracy(model, tokenizer, test_data)

    print("\n==============================")
    print("Fine-tuned model:", OUTPUT_DIR)
    print("Test accuracy:", round(acc, 4))
    print("==============================\n")

if __name__ == "__main__":
    main()


Loading data...
Loading tokenizer...
Tokenizing with answer-only supervision...


Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Loading base model (FP16, no quantization)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/tmp/ipython-input-3657802461.py:181: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.


Applying LoRA (rank-r tuning)...
trainable params: 3,145,728 || all params: 3,824,225,280 || trainable%: 0.0823
Training...


Step,Training Loss,Validation Loss
200,1.254700,1.232454
400,1.079400,1.151378
600,1.106700,1.117843
800,1.040400,1.107479
1000,1.073700,1.103891
1200,1.079700,1.090371
1400,1.098500,1.086354
1600,1.104900,1.082578
1800,1.061200,1.080750
2000,1.064000,1.081184


Step,Training Loss,Validation Loss
200,1.254700,1.232454
400,1.079400,1.151378
600,1.106700,1.117843
800,1.040400,1.107479
1000,1.073700,1.103891
1200,1.079700,1.090371
1400,1.098500,1.086354
1600,1.104900,1.082578
1800,1.061200,1.080750
2000,1.064000,1.081184


Saving adapter...

Evaluating on test set (yes/no accuracy)...

Fine-tuned model: ./phi3_lora_out
Test accuracy: 0.8963

